In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask, geometry_window
from rasterio.windows import Window
from shapely import make_valid
from shapely.validation import explain_validity

# ============================================================
# 1. Configuration
# ============================================================

BASE = Path("../data")

REF = Path(
    "../outputs/heatwave/reference_cells_all.csv"
)

SMOD = BASE / "smod_0514.shp"

ROOTS = {
    "ndvi": Path("../data/raw/ndvi"),
    "water": Path("../data/raw/water"),
}

OUT_BASE = Path(
    "../outputs/vegetation_water"
)

YEARS = list(range(2016, 2024))

NDVI_THR = 0.2
WATER_THR = 0.5

COVERAGE_MIN = 0.90

KEY = ["city", "cell_id", "year"]

METRICS = {
    "ndvi": [
        "ndvi_mean",
        "ndvi_std",
        "veg_coverage",
    ],
    "water": [
        "water_mean_prob",
        "water_std_prob",
        "water_coverage",
    ],
}

NAMES = {
    "Abidjan_IvoryCoast": "Abidjan",
    "AddisAbaba_Ethiopia": "Addis Ababa",
    "Alexandria_Egypt": "Alexandria",
    "Bamako_Mali": "Bamako",
    "Bangalore_India": "Bangalore",
    "Bangkok_Thailand": "Bangkok",
    "BeloHorizonte_Brazil": "Belo Horizonte",
    "Bogota_Colombia": "Bogota",
    "BuenosAires_Argentina": "Buenos Aires",
    "Cairo_Egypt": "Cairo",
    "Chennai_India": "Chennai",
    "Colombo_SriLanka": "Colombo",
    "DaresSalaam_Tanzania": "Dar es Salaam",
    "Delhi_India": "Delhi",
    "Dhaka_Bangladesh": "Dhaka",
    "Guayaquil_Ecuador": "Guayaquil",
    "HoChiMinh_Vietnam": "Ho Chi Minh City",
    "Hyderabad_India": "Hyderabad",
    "Jakarta_Indonesia": "Jakarta",
    "Johannesburg_SouthAfrica": "Johannesburg",
    "Karachi_Pakistan": "Karachi",
    "Kinshasa_DRC": "Kinshasa",
    "KualaLumpur_Malaysia": "Kuala Lumpur",
    "Lagos_Nigeria": "Lagos",
    "Lahore_Pakistan": "Lahore",
    "Lima_Peru": "Lima",
    "Manaus_Brazil": "Manaus",
    "Manila_Philippines": "Manila",
    "MexicoCity_Mexico": "Mexico City",
    "Mumbai_India": "Mumbai",
    "Nairobi_Kenya": "Nairobi",
    "PhnomPenh_Cambodia": "Phnom Penh",
    "Santiago_Chile": "Santiago",
    "SantoDomingo_DominicanRepublic": "Santo Domingo",
    "SaoPaulo_Brazil": "Sao Paulo",
}


# ============================================================
# 2. File and geometry utilities
# ============================================================

def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def stamp(path):
    s = Path(path).stat()
    return {
        "bytes": s.st_size,
        "mtime_ns": s.st_mtime_ns,
    }


def normalize(s):
    return "".join(
        c.lower() for c in s if c.isalnum()
    )


def source_index(kind):
    token = (
        "_Landsat_NDVI_"
        if kind == "ndvi"
        else "_DW_water_"
    )

    require(
        ROOTS[kind].is_dir(),
        f"Folder not found: {ROOTS[kind]}",
    )

    index = {}

    for p in ROOTS[kind].glob("*.tif"):
        if token not in p.stem:
            continue

        prefix, year = p.stem.rsplit(token, 1)

        require(
            year.isdigit(),
            f"Cannot parse year: {p}",
        )

        key = (normalize(prefix), int(year))

        require(
            key not in index,
            f"Duplicate files for the same city and year: {p}",
        )

        index[key] = p

    return index


def repair_polygon(geom, label, repairs):
    require(
        geom is not None and not geom.is_empty,
        f"Empty geometry: {label}",
    )

    if not geom.is_valid:
        reason = explain_validity(geom)
        fixed = make_valid(geom)

        require(
            fixed.geom_type in ["Polygon", "MultiPolygon"]
            and fixed.is_valid
            and not fixed.is_empty,
            f"Geometry repair failed: {label}",
        )

        delta = abs(fixed.area - geom.area)

        require(
            delta <= max(1e-12, abs(geom.area) * 1e-9)
            and np.allclose(
                fixed.bounds,
                geom.bounds,
                atol=1e-10,
                rtol=0,
            ),
            f"Repair changed the extent or area, needs separate review: {label}",
        )

        repairs.append({
            "name": label,
            "reason": reason,
            "area_difference_deg2": float(delta),
        })

        geom = fixed

        print(
            f"[Geometry repaired in memory only] {label}",
            flush=True,
        )

    require(
        geom.geom_type in ["Polygon", "MultiPolygon"],
        f"Not a polygon geometry: {label}",
    )

    return geom


def read_retry(src, window):
    for attempt in range(3):
        try:
            if attempt == 0:
                return src.read(
                    1,
                    window=window,
                    masked=True,
                )

            with rasterio.open(src.name) as reopened:
                return reopened.read(
                    1,
                    window=window,
                    masked=True,
                )

        except (OSError, rasterio.errors.RasterioIOError):
            if attempt == 2:
                raise

            time.sleep(2 * (attempt + 1))


# ============================================================
# 3. Metrics and coverage quality
# ============================================================

def summarize_values(
    values,
    study_pixels,
    outside_pixels,
    invalid_pixels,
    kind,
):
    names = METRICS[kind]
    n = len(values)

    frac = (
        n / study_pixels
        if study_pixels
        else np.nan
    )

    passed = bool(
        study_pixels > 0
        and frac >= COVERAGE_MIN
    )

    observed = dict.fromkeys(names, np.nan)

    if n:
        if kind == "ndvi":
            positive = values > NDVI_THR
        else:
            positive = values >= WATER_THR

        observed = dict(zip(
            names,
            [
                float(values.mean()),
                float(values.std(ddof=0)),
                float(positive.mean()),
            ],
        ))

    if study_pixels == 0:
        status = "no_study_pixels"
    elif n == 0:
        status = "no_valid_data"
    elif n == study_pixels:
        status = "complete"
    elif passed:
        status = "pass_with_small_gap"
    else:
        status = "insufficient_coverage"

    out = {
        f"{kind}_study_pixels": study_pixels,
        f"{kind}_valid_pixels": n,
        f"{kind}_missing_pixels": study_pixels - n,
        f"{kind}_outside_raster_pixels": outside_pixels,
        f"{kind}_missing_inside_raster_pixels": (
            study_pixels - outside_pixels - n
        ),
        f"{kind}_unmasked_invalid_value_pixels": (
            invalid_pixels
        ),
        f"{kind}_valid_frac": frac,
        f"{kind}_coverage_pass": passed,
        f"{kind}_coverage_complete": bool(
            study_pixels > 0 and n == study_pixels
        ),
        f"{kind}_quality_status": status,
    }

    for name, value in observed.items():
        out[name + "_observed"] = value
        out[name] = value if passed else np.nan

    return out


def calculate_region(src, geom, kind):
    w = geometry_window(
        src,
        [geom],
        boundless=True,
    )

    x0, y0, width, height = map(
        int,
        (w.col_off, w.row_off, w.width, w.height),
    )

    require(
        0 < width * height <= 25_000_000,
        f"Abnormal study window: {src.name}",
    )

    full = Window(x0, y0, width, height)

    inside = geometry_mask(
        [geom],
        out_shape=(height, width),
        transform=src.window_transform(full),
        invert=True,
        all_touched=False,
    )

    total = int(inside.sum())

    x1 = x0 + width
    y1 = y0 + height

    left = max(x0, 0)
    top = max(y0, 0)
    right = min(x1, src.width)
    bottom = min(y1, src.height)

    if right <= left or bottom <= top:
        return summarize_values(
            np.array([], dtype=float),
            total,
            total,
            0,
            kind,
        )

    submask = inside[
        top - y0:bottom - y0,
        left - x0:right - x0,
    ]

    n_inside_file = int(submask.sum())

    if not n_inside_file:
        return summarize_values(
            np.array([], dtype=float),
            total,
            total,
            0,
            kind,
        )

    arr = read_retry(
        src,
        Window(
            left,
            top,
            right - left,
            bottom - top,
        ),
    )

    data = np.asarray(
        arr.data,
        dtype="float64",
    )

    unmasked = (
        submask & ~np.ma.getmaskarray(arr)
    )

    lo = -1 if kind == "ndvi" else 0

    valid = (
        unmasked
        & np.isfinite(data)
        & (data != -9999)
        & (data >= lo)
        & (data <= 1)
    )

    invalid_count = int(
        (unmasked & ~valid).sum()
    )

    return summarize_values(
        data[valid],
        total,
        total - n_inside_file,
        invalid_count,
        kind,
    )


# ============================================================
# 4. Input pre-check, computation and saving
# ============================================================

def run_green_water():
    require(
        0 < COVERAGE_MIN <= 1,
        "Coverage threshold should be in (0,1].",
    )

    sources = []
    tasks = []
    cells_by_city = {}
    repairs = []

    def register(path, **extra):
        require(
            path.is_file(),
            f"Missing input: {path}",
        )

        sources.append({
            "path": str(path),
            **stamp(path),
            **extra,
        })

    register(REF, role="research_points")

    for ext in [".shp", ".shx", ".dbf", ".prj"]:
        register(
            SMOD.with_suffix(ext),
            role="SMOD_boundary",
        )

    pts = pd.read_csv(REF)[
        ["city", "cell_id", "lat", "lon"]
    ]

    require(
        len(pts) == 652
        and set(pts.city) == set(NAMES)
        and not pts.duplicated(
            ["city", "cell_id"]
        ).any(),
        "Abnormal list of 35 cities and 652 grid cells.",
    )

    boundary = gpd.read_file(SMOD)

    require(
        boundary.crs is not None,
        "SMOD has no CRS.",
    )

    boundary = boundary.to_crs(4326)

    for i, r in boundary.iterrows():
        boundary.at[i, "geometry"] = repair_polygon(
            r.geometry,
            str(r["name"]),
            repairs,
        )

    indexes = {
        kind: source_index(kind)
        for kind in ROOTS
    }

    for city, name in NAMES.items():
        selected = (
            pts[pts.city.eq(city)]
            .sort_values("cell_id")
        )

        b = boundary[boundary.name.eq(name)]

        require(
            len(b) == 1,
            f"Cannot match SMOD uniquely: {city}",
        )

        urban = b.geometry.iloc[0]

        gp = (
            BASE / "grids"
            / f"{city}_era5_grid.gpkg"
        )

        register(gp, role="original_grid")

        grid = gpd.read_file(gp)

        require(
            grid.crs is not None
            and grid.cell_id.is_unique,
            f"Abnormal grid: {city}",
        )

        grid = (
            grid.to_crs(4326)
            .set_index("cell_id")
            .loc[selected.cell_id]
        )

        require(
            np.allclose(
                grid.lat_center,
                selected.lat,
                atol=1e-6,
                rtol=0,
            )
            and np.allclose(
                grid.lon_center,
                selected.lon,
                atol=1e-6,
                rtol=0,
            ),
            f"Coordinates do not match: {city}",
        )

        cells = []

        for cid, r in grid.iterrows():
            cell_geom = repair_polygon(
                r.geometry,
                f"{city}/{cid}",
                repairs,
            )

            geom = cell_geom.intersection(urban)

            require(
                not geom.is_empty
                and geom.area > 0
                and geom.is_valid,
                f"Abnormal study intersection: {city}/{cid}",
            )

            cells.append((
                int(cid),
                float(r.lat_center),
                float(r.lon_center),
                geom,
            ))

        cells_by_city[city] = cells

        for year in YEARS:
            paths = {}

            for kind in ROOTS:
                path = indexes[kind].get(
                    (normalize(name), year)
                )

                require(
                    path is not None,
                    f"Missing {kind} file: {city} {year}",
                )

                register(
                    path,
                    role=kind,
                    city=city,
                    year=year,
                )

                with rasterio.open(path) as src:
                    t = src.transform

                    expected_band = (
                        "NDVI"
                        if kind == "ndvi"
                        else "water_prob"
                    )

                    require(
                        src.count == 1
                        and src.descriptions[0] == expected_band,
                        f"Not a verified single-band product: {path}",
                    )

                    require(
                        src.crs is not None
                        and src.crs.to_epsg() == 4326
                        and t.a > 0
                        and t.e < 0
                        and t.b == 0
                        and t.d == 0,
                        f"Abnormal CRS or orientation: {path}",
                    )

                    require(
                        np.issubdtype(
                            np.dtype(src.dtypes[0]),
                            np.floating,
                        )
                        and src.scales[0] == 1
                        and src.offsets[0] == 0,
                        "Not a verified float raw-value product, "
                        f"scaling must not be guessed: {path}",
                    )

                    sources[-1].update(
                        crs=str(src.crs),
                        width=src.width,
                        height=src.height,
                        transform=str(tuple(t)),
                        band=expected_band,
                    )

                paths[kind] = path

            tasks.append((city, year, paths))

        print(
            f"[Pre-check] {city}: "
            "NDVI and water files complete for all 8 years",
            flush=True,
        )

    out = OUT_BASE

    (out / "by_city_year").mkdir(
        parents=True,
        exist_ok=True,
    )

    summary = {
        "status": "running",
        "cities": 35,
        "points": 652,
        "years": YEARS,
        "coverage_min": COVERAGE_MIN,
        "geometry_repairs": repairs,
        "panels_modified": False,
    }

    def save_summary():
        (out / "summary.json").write_text(
            json.dumps(
                summary,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

    def save(frame, name):
        path = out / name

        frame.to_csv(
            path,
            index=False,
            encoding="utf-8-sig",
        )

        back = pd.read_csv(path)

        pd.testing.assert_frame_equal(
            frame.reset_index(drop=True),
            back,
            check_dtype=False,
            check_exact=False,
            rtol=1e-12,
            atol=1e-10,
        )

    save_summary()

    pd.DataFrame(sources).to_csv(
        out / "input_manifest.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print("Output folder:", out, flush=True)

    records = []

    try:
        for city, year, paths in tasks:
            batch = [
                {
                    "city": city,
                    "cell_id": cid,
                    "year": year,
                    "lat": lat,
                    "lon": lon,
                }
                for cid, lat, lon, _
                in cells_by_city[city]
            ]

            for kind, path in paths.items():
                with rasterio.open(path) as src:
                    for rec, (_, _, _, geom) in zip(
                        batch,
                        cells_by_city[city],
                    ):
                        rec.update(
                            calculate_region(
                                src, geom, kind
                            )
                        )

                        rec[kind + "_source_path"] = str(path)

            for rec in batch:
                rec["green_water_coverage_pass"] = (
                    rec["ndvi_coverage_pass"]
                    and rec["water_coverage_pass"]
                )

                rec["needs_review"] = not (
                    rec["ndvi_coverage_complete"]
                    and rec["water_coverage_complete"]
                )

            save(
                pd.DataFrame(batch),
                f"by_city_year/{city}_{year}.csv",
            )

            records.extend(batch)

            summary["rows_finished"] = len(records)
            save_summary()

            ndvi_pass = sum(
                r["ndvi_coverage_pass"] for r in batch
            )
            water_pass = sum(
                r["water_coverage_pass"] for r in batch
            )

            print(
                f"[{len(records)}/5216] {city} {year}: "
                f"NDVI passed {ndvi_pass}/{len(batch)}; "
                f"water passed {water_pass}/{len(batch)}",
                flush=True,
            )

        result = (
            pd.DataFrame(records)
            .sort_values(KEY)
            .reset_index(drop=True)
        )

        require(
            len(result) == 5216
            and not result.duplicated(KEY).any(),
            "Abnormal output row count or unique key.",
        )

        for kind, metrics in METRICS.items():
            passed = result[kind + "_coverage_pass"]

            require(
                result.loc[
                    ~passed, metrics
                ].isna().all().all(),
                "Final metrics wrongly retained for low-coverage records.",
            )

            require(
                np.isfinite(
                    result.loc[passed, metrics]
                ).all().all(),
                "Passing records contain invalid metrics.",
            )

        for s in sources:
            require(
                stamp(s["path"])
                == {
                    k: s[k]
                    for k in ["bytes", "mtime_ns"]
                },
                f"Input changed during the run: {s['path']}",
            )

        save(
            result,
            "green_water_smod_all_2016_2023.csv",
        )

        save(
            result.loc[result.needs_review],
            "rows_requiring_review.csv",
        )

        quality = result.groupby(
            ["city", "year"],
            as_index=False,
        ).agg(
            rows=("cell_id", "size"),
            ndvi_pass_rows=(
                "ndvi_coverage_pass", "sum"
            ),
            water_pass_rows=(
                "water_coverage_pass", "sum"
            ),
            both_pass_rows=(
                "green_water_coverage_pass", "sum"
            ),
            ndvi_min_valid_frac=(
                "ndvi_valid_frac", "min"
            ),
            water_min_valid_frac=(
                "water_valid_frac", "min"
            ),
        )

        save(quality, "city_year_quality.csv")

        summary.update(
            status="completed",
            rows=len(result),
            ndvi_pass_rows=int(
                result.ndvi_coverage_pass.sum()
            ),
            water_pass_rows=int(
                result.water_coverage_pass.sum()
            ),
            both_pass_rows=int(
                result.green_water_coverage_pass.sum()
            ),
            review_rows=int(
                result.needs_review.sum()
            ),
        )

        save_summary()

        print("Output folder:", out)

        return out, result, quality

    except (Exception, KeyboardInterrupt) as exc:
        summary.update(
            status="failed",
            error=repr(exc),
            rows_finished=len(records),
        )

        save_summary()
        raise


# ============================================================
# 5. Run
# ============================================================

(
    GREEN_WATER_DIR,
    GREEN_WATER_METRICS,
    GREEN_WATER_QUALITY,
) = run_green_water()

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

GREEN_WATER_RUN = Path("../outputs/vegetation_water")

MIN_DONOR_YEARS = 1

KEY = ["city", "cell_id", "year"]
WATER = ["water_mean_prob", "water_std_prob", "water_coverage"]


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def fill_from_other_years(df, kind, metrics):
    passed = df[f"{kind}_coverage_pass"].astype(bool)
    require(df.loc[~passed, metrics].isna().all().all(), f"{kind}: final columns of rows below the threshold should be NaN")
    require(df.loc[passed, metrics].notna().all().all(), f"{kind}: final columns of rows meeting the threshold should not be NaN")

    donors = df.loc[passed].groupby(["city", "cell_id"])
    means = donors[metrics].mean()
    n_donor = donors.size().rename(f"{kind}_fill_donor_years")

    target = df.loc[~passed, ["city", "cell_id"]].join(means, on=["city", "cell_id"]).join(
        n_donor, on=["city", "cell_id"])
    target[f"{kind}_fill_donor_years"] = target[f"{kind}_fill_donor_years"].fillna(0).astype(int)
    ok = target[f"{kind}_fill_donor_years"] >= MIN_DONOR_YEARS

    df[f"{kind}_filled_from_other_years"] = False
    df[f"{kind}_fill_donor_years"] = 0
    idx = target.index[ok]
    df.loc[idx, metrics] = target.loc[idx, metrics].to_numpy()
    df.loc[idx, f"{kind}_filled_from_other_years"] = True
    df.loc[target.index, f"{kind}_fill_donor_years"] = target[f"{kind}_fill_donor_years"]
    return int(ok.sum()), int((~ok).sum())


def run_fill():
    src = GREEN_WATER_RUN / "green_water_smod_all_2016_2023.csv"
    require(src.is_file(), f"Not found: {src}")
    df = pd.read_csv(src)
    require(not df.duplicated(KEY).any(), "Duplicate keys")

    report = {"source": str(src), "min_donor_years": MIN_DONOR_YEARS}
    n_fail = int((~df["water_coverage_pass"].astype(bool)).sum())
    filled, left = fill_from_other_years(df, "water", WATER)
    report["water"] = dict(rows_below_threshold=n_fail, filled=filled, still_nan=left)

    out = GREEN_WATER_RUN / "green_water_smod_all_2016_2023_waterfilled.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig", float_format="%.17g")
    back = pd.read_csv(out, float_precision="round_trip")
    pd.testing.assert_frame_equal(df.reset_index(drop=True), back, check_dtype=False, check_exact=True)
    (GREEN_WATER_RUN / "water_fill_report.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

    print("Output:", out)
    return df


GREEN_WATER_FILLED = run_fill()

In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask, geometry_window
from rasterio.windows import Window
from shapely import make_valid
from shapely.validation import explain_validity

# ============================================================
# 1. Configuration
# ============================================================

BASE = Path("../data")
SOURCE_DIR = Path("../data/raw/impervious")
SMOD = BASE / "smod_0514.shp"

REF = Path(
    "../outputs/heatwave/reference_cells_all.csv"
)

OUT_BASE = Path(
    "../outputs/impervious"
)

YEARS = list(range(2016, 2024))

COVERAGE_MIN = 0.90

BUILT_THRESHOLD = 0.5

KEY = ["city", "cell_id", "year"]

METRICS = [
    "isa_mean_prob",
    "isa_std_prob",
    "isa_coverage",
]

NAMES = {
    "Abidjan_IvoryCoast": "Abidjan",
    "AddisAbaba_Ethiopia": "Addis Ababa",
    "Alexandria_Egypt": "Alexandria",
    "Bamako_Mali": "Bamako",
    "Bangalore_India": "Bangalore",
    "Bangkok_Thailand": "Bangkok",
    "BeloHorizonte_Brazil": "Belo Horizonte",
    "Bogota_Colombia": "Bogota",
    "BuenosAires_Argentina": "Buenos Aires",
    "Cairo_Egypt": "Cairo",
    "Chennai_India": "Chennai",
    "Colombo_SriLanka": "Colombo",
    "DaresSalaam_Tanzania": "Dar es Salaam",
    "Delhi_India": "Delhi",
    "Dhaka_Bangladesh": "Dhaka",
    "Guayaquil_Ecuador": "Guayaquil",
    "HoChiMinh_Vietnam": "Ho Chi Minh City",
    "Hyderabad_India": "Hyderabad",
    "Jakarta_Indonesia": "Jakarta",
    "Johannesburg_SouthAfrica": "Johannesburg",
    "Karachi_Pakistan": "Karachi",
    "Kinshasa_DRC": "Kinshasa",
    "KualaLumpur_Malaysia": "Kuala Lumpur",
    "Lagos_Nigeria": "Lagos",
    "Lahore_Pakistan": "Lahore",
    "Lima_Peru": "Lima",
    "Manaus_Brazil": "Manaus",
    "Manila_Philippines": "Manila",
    "MexicoCity_Mexico": "Mexico City",
    "Mumbai_India": "Mumbai",
    "Nairobi_Kenya": "Nairobi",
    "PhnomPenh_Cambodia": "Phnom Penh",
    "Santiago_Chile": "Santiago",
    "SantoDomingo_DominicanRepublic": "Santo Domingo",
    "SaoPaulo_Brazil": "Sao Paulo",
}


# ============================================================
# 2. Basic utilities
# ============================================================

def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def stamp(path):
    s = Path(path).stat()
    return {
        "bytes": s.st_size,
        "mtime_ns": s.st_mtime_ns,
    }


def norm(s):
    return "".join(
        c.lower() for c in s if c.isalnum()
    )


def repair(geom, label, log):
    require(
        geom is not None and not geom.is_empty,
        f"Empty geometry: {label}",
    )

    if not geom.is_valid:
        fixed = make_valid(geom)
        delta = abs(fixed.area - geom.area)

        require(
            fixed.is_valid
            and not fixed.is_empty
            and fixed.geom_type in ["Polygon", "MultiPolygon"]
            and delta <= max(1e-12, abs(geom.area) * 1e-9)
            and np.allclose(
                fixed.bounds,
                geom.bounds,
                atol=1e-10,
                rtol=0,
            ),
            f"Geometry repair changed the extent or failed, needs review: {label}",
        )

        log.append({
            "name": label,
            "reason": explain_validity(geom),
            "area_difference_deg2": float(delta),
        })

        geom = fixed
        print(f"[Repaired in memory only] {label}", flush=True)

    require(
        geom.geom_type in ["Polygon", "MultiPolygon"],
        f"Not a polygon geometry: {label}",
    )

    return geom


def read_retry(src, window):
    for attempt in range(3):
        try:
            if attempt == 0:
                return src.read(
                    1,
                    window=window,
                    masked=True,
                )

            with rasterio.open(src.name) as retry:
                return retry.read(
                    1,
                    window=window,
                    masked=True,
                )

        except (OSError, rasterio.errors.RasterioIOError):
            if attempt == 2:
                raise

            time.sleep(2 * (attempt + 1))


# ============================================================
# 3. Single grid-cell computation
# ============================================================

def calculate_cell(src, geom):
    w = geometry_window(
        src,
        [geom],
        boundless=True,
    )

    x, y, width, height = map(
        int,
        (w.col_off, w.row_off, w.width, w.height),
    )

    require(
        0 < width * height <= 25_000_000,
        f"Abnormal study window: {src.name}",
    )

    full = Window(x, y, width, height)

    inside = geometry_mask(
        [geom],
        out_shape=(height, width),
        transform=src.window_transform(full),
        invert=True,
        all_touched=False,
    )

    total = int(inside.sum())

    left = max(0, x)
    top = max(0, y)
    right = min(src.width, x + width)
    bottom = min(src.height, y + height)

    in_file = 0
    masked = 0
    nonfinite = 0
    sentinel = 0
    out_of_range = 0

    vals = np.array([], dtype="float64")

    if right > left and bottom > top:
        region = inside[
            top - y:bottom - y,
            left - x:right - x,
        ]

        in_file = int(region.sum())

        if in_file:
            arr = read_retry(
                src,
                Window(
                    left,
                    top,
                    right - left,
                    bottom - top,
                ),
            )

            data = np.asarray(
                arr.data,
                dtype="float64",
            )

            mask = np.ma.getmaskarray(arr)

            unmasked = region & ~mask
            finite = unmasked & np.isfinite(data)

            masked = int((region & mask).sum())

            nonfinite = int(
                (unmasked & ~np.isfinite(data)).sum()
            )

            sentinel = int(
                (finite & (data == -9999)).sum()
            )

            outside_range = (
                finite
                & (data != -9999)
                & ((data < 0) | (data > 1))
            )

            out_of_range = int(outside_range.sum())

            valid = (
                finite
                & (data >= 0)
                & (data <= 1)
            )

            vals = data[valid]

    n = len(vals)
    outside = total - in_file

    require(
        total
        == (
            n
            + outside
            + masked
            + nonfinite
            + sentinel
            + out_of_range
        ),
        "Pixel classification counts are not conserved.",
    )

    frac = n / total if total else np.nan

    passed = bool(
        total > 0 and frac >= COVERAGE_MIN
    )

    observed = [np.nan] * 3

    if n:
        observed = [
            float(vals.mean()),
            float(vals.std(ddof=0)),
            float(np.mean(vals >= BUILT_THRESHOLD)),
        ]

    if not total:
        status = "no_study_pixels"
    elif not n:
        status = "no_valid_data"
    elif n == total:
        status = "complete"
    elif passed:
        status = "pass_with_gap"
    else:
        status = "insufficient_coverage"

    rec = {
        "isa_study_pixels": total,
        "isa_valid_pixels": n,
        "isa_missing_pixels": total - n,
        "isa_outside_raster_pixels": outside,
        "isa_missing_inside_raster_pixels": in_file - n,
        "isa_masked_pixels": masked,
        "isa_nonfinite_pixels": nonfinite,
        "isa_unmasked_minus9999_pixels": sentinel,
        "isa_out_of_range_pixels": out_of_range,
        "isa_valid_frac": frac,
        "isa_coverage_pass": passed,
        "isa_coverage_complete": bool(
            total > 0 and n == total
        ),
        "isa_quality_status": status,
        "isa_imputed": False,
    }

    for name, value in zip(METRICS, observed):
        rec[name + "_observed"] = value
        rec[name] = value if passed else np.nan

    rec["isa_needs_review"] = (
        not rec["isa_coverage_complete"]
    )

    return rec


# ============================================================
# 4. Input pre-check
# ============================================================

def run_isa():
    require(
        0 < COVERAGE_MIN <= 1,
        "Coverage threshold should be in (0,1].",
    )

    require(
        SOURCE_DIR.is_dir(),
        f"Source folder not found: {SOURCE_DIR}",
    )

    sources = []
    tasks = []
    repairs = []

    def register(path, **info):
        require(
            path.is_file(),
            f"Missing input: {path}",
        )

        sources.append({
            "path": str(path),
            **stamp(path),
            **info,
        })

    register(REF, role="research_points")

    for ext in [".shp", ".shx", ".dbf", ".prj"]:
        register(
            SMOD.with_suffix(ext),
            role="SMOD_boundary",
        )

    points = pd.read_csv(REF)[
        ["city", "cell_id", "lat", "lon"]
    ]

    require(
        len(points) == 652
        and set(points.city) == set(NAMES)
        and not points.duplicated(
            ["city", "cell_id"]
        ).any(),
        "Abnormal study grid-cell list.",
    )

    boundary = gpd.read_file(SMOD)

    require(
        boundary.crs is not None,
        "SMOD has no CRS.",
    )

    boundary = boundary.to_crs(4326)

    for i, r in boundary.iterrows():
        boundary.at[i, "geometry"] = repair(
            r.geometry,
            str(r["name"]),
            repairs,
        )

    index = {}

    for path in SOURCE_DIR.glob("*.tif"):
        if "_DW_built_" not in path.stem:
            continue

        prefix, year = path.stem.rsplit(
            "_DW_built_", 1
        )

        require(
            year.isdigit(),
            f"Cannot parse year: {path}",
        )

        key = (norm(prefix), int(year))

        require(
            key not in index,
            f"Duplicate files for the same city and year: {path}",
        )

        index[key] = path

    for city, name in NAMES.items():
        urban = boundary[
            boundary.name.eq(name)
        ]

        require(
            len(urban) == 1,
            f"SMOD cannot be matched uniquely: {city}",
        )

        gp = (
            BASE / "grids"
            / f"{city}_era5_grid.gpkg"
        )

        register(gp, role="original_grid")

        grid = gpd.read_file(gp)

        require(
            grid.crs is not None
            and grid.cell_id.is_unique,
            f"Abnormal grid: {city}",
        )

        selected = (
            points[points.city.eq(city)]
            .sort_values("cell_id")
        )

        grid = (
            grid.to_crs(4326)
            .set_index("cell_id")
            .loc[selected.cell_id]
        )

        require(
            np.allclose(
                grid.lat_center,
                selected.lat,
                atol=1e-6,
                rtol=0,
            )
            and np.allclose(
                grid.lon_center,
                selected.lon,
                atol=1e-6,
                rtol=0,
            ),
            f"Inconsistent coordinates: {city}",
        )

        cells = []

        for cid, row in grid.iterrows():
            cell_geom = repair(
                row.geometry,
                f"{city}/{cid}",
                repairs,
            )

            geom = cell_geom.intersection(
                urban.geometry.iloc[0]
            )

            require(
                not geom.is_empty
                and geom.area > 0
                and geom.is_valid,
                f"Invalid study intersection: {city}/{cid}",
            )

            cells.append((
                int(cid),
                float(row.lat_center),
                float(row.lon_center),
                geom,
            ))

        signature = None

        for year in YEARS:
            path = index.get(
                (norm(name), year)
            )

            require(
                path is not None,
                f"Missing ISA: {city} {year}",
            )

            register(
                path,
                role="ISA_built_probability",
                city=city,
                year=year,
            )

            with rasterio.open(path) as src:
                t = src.transform

                require(
                    src.count == 1
                    and src.descriptions[0] == "built_prob",
                    f"Not the expected single band built_prob: {path}",
                )

                require(
                    src.crs is not None
                    and src.crs.to_epsg() == 4326
                    and t.a > 0
                    and t.e < 0
                    and t.b == 0
                    and t.d == 0,
                    f"Abnormal CRS or orientation: {path}",
                )

                require(
                    np.issubdtype(
                        np.dtype(src.dtypes[0]),
                        np.floating,
                    )
                    and src.scales[0] == 1
                    and src.offsets[0] == 0,
                    "Not a verified float raw-value product, "
                    f"scaling must not be guessed: {path}",
                )

                sig = (
                    tuple(t),
                    src.width,
                    src.height,
                )

                if signature is None:
                    signature = sig

                require(
                    sig == signature,
                    f"{city}: the grid changed between years, review first.",
                )

                sources[-1].update(
                    width=src.width,
                    height=src.height,
                    transform=str(tuple(t)),
                    crs=str(src.crs),
                    band="built_prob",
                )

            tasks.append(
                (city, year, path, cells)
            )

        print(
            f"[Pre-check] {city}: files complete for all 8 years, grid consistent",
            flush=True,
        )

    # ========================================================
    # 5. Create a new output folder
    # ========================================================

    out = OUT_BASE

    (out / "by_city_year").mkdir(
        parents=True,
        exist_ok=True,
    )

    summary = {
        "status": "running",
        "cities": 35,
        "points": 652,
        "years": YEARS,
        "coverage_min": COVERAGE_MIN,
        "geometry_repairs": repairs,
    }

    def save_summary():
        (out / "summary.json").write_text(
            json.dumps(
                summary,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

    def save(frame, filename):
        path = out / filename

        frame.to_csv(
            path,
            index=False,
            encoding="utf-8-sig",
        )

        back = pd.read_csv(path)

        pd.testing.assert_frame_equal(
            frame.reset_index(drop=True),
            back,
            check_dtype=False,
            check_exact=False,
            rtol=1e-12,
            atol=1e-10,
        )

    save_summary()

    pd.DataFrame(sources).to_csv(
        out / "input_manifest.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print("Output folder:", out, flush=True)

    # ========================================================
    # 6. Computation by city and year
    # ========================================================

    records = []

    try:
        for task_number, task in enumerate(tasks, 1):
            city, year, path, cells = task
            batch = []

            with rasterio.open(path) as src:
                for cid, lat, lon, geom in cells:
                    rec = {
                        "city": city,
                        "cell_id": cid,
                        "year": year,
                        "lat": lat,
                        "lon": lon,
                        "isa_source_path": str(path),
                    }

                    rec.update(
                        calculate_cell(src, geom)
                    )

                    batch.append(rec)

            save(
                pd.DataFrame(batch),
                f"by_city_year/{city}_{year}.csv",
            )

            records.extend(batch)

            summary.update(
                rows_finished=len(records),
                city_years_finished=task_number,
            )

            save_summary()

            n_pass = sum(
                r["isa_coverage_pass"]
                for r in batch
            )

            print(
                f"[{len(records)}/5216] {city} {year}: "
                f"coverage threshold met {n_pass}/{len(batch)}",
                flush=True,
            )

        # ====================================================
        # 7. Validation and summary
        # ====================================================

        result = (
            pd.DataFrame(records)
            .sort_values(KEY)
            .reset_index(drop=True)
        )

        require(
            len(result) == 5216
            and not result.duplicated(KEY).any(),
            "Abnormal output row count or key.",
        )

        require(
            result.groupby(
                ["city", "cell_id"]
            ).isa_study_pixels.nunique().eq(1).all(),
            "Study-pixel denominator changes across years.",
        )

        require(
            result.loc[
                ~result.isa_coverage_pass,
                METRICS,
            ].isna().all().all(),
            "Final metrics wrongly retained for low-coverage records.",
        )

        require(
            np.isfinite(
                result.loc[
                    result.isa_coverage_pass,
                    METRICS,
                ]
            ).all().all(),
            "Missing metrics in records that meet the coverage threshold.",
        )

        require(
            result.isa_mean_prob_observed
            .dropna().between(0, 1).all()
            and result.isa_coverage_observed
            .dropna().between(0, 1).all(),
            "Probability or coverage out of range.",
        )

        for s in sources:
            require(
                stamp(s["path"])
                == {
                    k: s[k]
                    for k in ["bytes", "mtime_ns"]
                },
                f"Input changed during the run: {s['path']}",
            )

        save(
            result,
            "isa_smod_all_2016_2023.csv",
        )

        save(
            result.loc[result.isa_needs_review],
            "isa_rows_requiring_review.csv",
        )

        save(
            result.loc[~result.isa_coverage_pass],
            "isa_rows_below_threshold.csv",
        )

        quality = result.groupby(
            ["city", "year"],
            as_index=False,
        ).agg(
            rows=("cell_id", "size"),
            pass_rows=("isa_coverage_pass", "sum"),
            complete_rows=(
                "isa_coverage_complete", "sum"
            ),
            min_valid_frac=("isa_valid_frac", "min"),
        )

        save(
            quality,
            "city_year_quality.csv",
        )

        summary.update(
            status="completed",
            rows=len(result),
            pass_rows=int(
                result.isa_coverage_pass.sum()
            ),
            complete_rows=int(
                result.isa_coverage_complete.sum()
            ),
            no_valid_data_rows=int(
                result.isa_valid_pixels.eq(0).sum()
            ),
            invalid_out_of_range_pixels=int(
                result.isa_out_of_range_pixels.sum()
            ),
        )

        save_summary()

        print("Output folder:", out)

        return out, result, quality

    except (Exception, KeyboardInterrupt) as exc:
        summary.update(
            status="failed",
            error=repr(exc),
            rows_finished=len(records),
        )

        save_summary()
        raise


# ============================================================
# 8. Execution
# ============================================================

(
    ISA_OUTPUT_DIR,
    ISA_METRICS,
    ISA_QUALITY,
) = run_isa()

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ISA_RUN = Path("../outputs/impervious")

KEY = ["city", "cell_id", "year"]
METRICS = ["isa_mean_prob", "isa_std_prob", "isa_coverage"]


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def fill_cell(g):
    g = g.sort_values("year").copy()
    passed = g["isa_coverage_pass"].to_numpy(bool)
    years = g["year"].to_numpy(float)
    py = years[passed]
    method = np.where(passed, "observed", "")
    if (~passed).any():
        inside = (years > py.min()) & (years < py.max())
        method = np.where(~passed & inside, "interpolated", method)
        method = np.where(~passed & ~inside, "nearest_year", method)
        for c in METRICS:
            g[c] = np.interp(years, py, g[c].to_numpy(float)[passed])
    g["isa_fill_method"] = method
    g["isa_fill_gap_years"] = np.abs(years[:, None] - py[None, :]).min(axis=1).astype(int)
    return g


def run_isa_fill():
    src = ISA_RUN / "isa_smod_all_2016_2023.csv"
    require(src.is_file(), f"Not found: {src}")
    df = pd.read_csv(src)
    require(not df.duplicated(KEY).any(), "Duplicate keys")
    passed = df["isa_coverage_pass"].astype(bool)
    df["isa_coverage_pass"] = passed
    require(df.loc[~passed, METRICS].isna().all().all(), "Final columns of rows below the threshold should be NaN")
    require(df.loc[passed, METRICS].notna().all().all(), "Final columns of rows meeting the threshold should not be NaN")
    n_pass = df.groupby(["city", "cell_id"])["isa_coverage_pass"].sum()
    require((n_pass > 0).all(), f"Some grid cells fail the coverage threshold in all 8 years and cannot be filled: {n_pass[n_pass == 0].index.tolist()}")

    original = df.set_index(KEY)[METRICS].copy()
    out = pd.concat([fill_cell(g) for _, g in df.groupby(["city", "cell_id"], sort=False)])
    out = out.sort_values(KEY).reset_index(drop=True)
    out["isa_filled"] = out["isa_fill_method"] != "observed"
    out["isa_imputed"] = out["isa_filled"]

    chk = out.set_index(KEY)
    ok_rows = original.index[passed.to_numpy()]
    require(np.array_equal(chk.loc[ok_rows, METRICS].to_numpy(), original.loc[ok_rows, METRICS].to_numpy()),
            "Rows meeting the threshold were modified")
    require(out[METRICS].notna().all().all(), "NaN remains after filling")

    dst = ISA_RUN / "isa_smod_all_2016_2023_filled.csv"
    out.to_csv(dst, index=False, encoding="utf-8-sig", float_format="%.17g")
    back = pd.read_csv(dst, float_precision="round_trip")
    pd.testing.assert_frame_equal(out, back, check_dtype=False, check_exact=True)

    report = {
        "source": str(src),
        "rows": int(len(out)),
        "rows_below_threshold": int((~passed).sum()),
        "fill_method_counts": out["isa_fill_method"].value_counts().to_dict(),
        "max_gap_years": int(out["isa_fill_gap_years"].max()),
    }
    (ISA_RUN / "isa_fill_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

    print("Output:", dst)
    return out


ISA_FILLED = run_isa_fill()